# Topic 5 — Advanced RAG: Practice Notebook

Books B01 and B02 covered *basic* RAG: embed chunks once, embed the query,
take the top-k by cosine similarity, stuff them in a prompt. That pipeline is
a great starting point and a surprisingly common failure point in production.
This notebook builds a small corpus drawn from this repo's own material
(tokenization concepts from B01, RLHF/DPO/GANs from B02, LangGraph/MCP
internals from B03 Topics 1-3) and uses it to *empirically* find where naive
RAG breaks, then fixes those breaks one technique at a time.

Sections:

1. The Limits of Naive RAG — find two real failure modes on our corpus.
2. Hybrid Search — BM25 + dense embeddings combined with Reciprocal Rank
   Fusion (RRF), fixing failure mode 1.
3. Cross-Encoder Reranking — a second-pass model that sharpens hybrid
   search's narrow margins into decisive rankings.
4. Query Transformation — multi-query decomposition (fixing failure mode 2)
   and HyDE (Hypothetical Document Embeddings).
5. Agentic RAG (exercise) — give an LLM a `retrieve` tool from Topic 4's
   ReAct pattern and let it decide when and how often to call it.
6. GraphRAG (exercise) — represent the corpus as a knowledge graph and
   answer multi-hop questions by graph traversal instead of chunk retrieval.

Sections 1-4 use a real local embedding model (`all-MiniLM-L6-v2`) and a real
local cross-encoder reranker (`cross-encoder/ms-marco-MiniLM-L-6-v2`) via
`sentence-transformers` — both run on CPU, downloaded once from Hugging Face,
fully deterministic. Any *generative* LLM step (query decomposition, HyDE,
the agent's tool-calling loop) uses `GenericFakeChatModel` with scripted
responses, exactly like Topics 1-4, so the whole notebook runs offline.

In [ ]:
import json

import numpy as np
import networkx as nx

from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

from typing import TypedDict, Annotated
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.tools import tool

from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

print("Imports OK.")

## 1. The Limits of Naive RAG

**Naive RAG** embeds every document chunk once with a *bi-encoder* (a model
that turns text into a fixed-size vector independently, with no knowledge of
the query), embeds the user's query the same way, and ranks chunks by cosine
similarity. It's cheap — the corpus embeddings are precomputed, so a query
only costs one embedding call plus a dot product against every stored vector.

The corpus below mixes documents from several topics already covered in this
repo, on purpose: a real RAG system over your own notes would look exactly
like this — short, topically-related-but-distinct paragraphs.

In [ ]:
CORPUS = {
    "bpe": "Byte Pair Encoding (BPE) is a subword tokenization algorithm that iteratively merges the most frequent pair of adjacent symbols into a new token.",
    "gpt2_tokenizer": "GPT-2 uses a BPE tokenizer with a vocabulary of 50,257 tokens to convert raw text into integer token IDs.",
    "self_attention": "Self-attention computes a weighted sum over all tokens in a sequence, where the weights come from query-key dot products.",
    "transformer": "The Transformer architecture, introduced in 'Attention Is All You Need' (2017), relies entirely on attention and removes recurrence.",
    "langgraph": "LangGraph is a library for building stateful, graph-based LLM applications using nodes and edges.",
    "checkpointer": "A checkpointer in LangGraph persists the state of a graph after every step, identified by a thread_id.",
    "tools_condition": "The tools_condition function in LangGraph checks whether the latest AIMessage contains tool_calls and routes to the tools node if so, or to END otherwise.",
    "mcp": "The Model Context Protocol (MCP) standardizes how LLM applications connect to external tools, data, and prompts via JSON-RPC.",
    "tool_annotations": "MCP's ToolAnnotations type includes readOnlyHint, destructiveHint, idempotentHint, and openWorldHint fields that describe a tool's side effects.",
    "rlhf": "Reinforcement Learning from Human Feedback (RLHF) fine-tunes a language model using a reward model trained on human preference data.",
    "dpo": "Direct Preference Optimization (DPO) optimizes a policy directly on preference pairs, without training a separate reward model.",
    "gan": "Generative Adversarial Networks (GANs) consist of a generator and a discriminator trained in an adversarial min-max game.",
    "rag": "Retrieval-Augmented Generation (RAG) combines a retriever that fetches relevant documents with a generator that conditions on those documents.",
    "bm25_doc": "BM25 is a sparse, keyword-based ranking function built on term frequency and inverse document frequency.",
}

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

doc_keys = list(CORPUS.keys())
doc_texts = list(CORPUS.values())
doc_embeddings = embedding_model.encode(doc_texts, normalize_embeddings=True)

print(f"Corpus: {len(CORPUS)} documents")
print(f"Embedding matrix shape: {doc_embeddings.shape}")  # (num_docs, 384)

In [ ]:
def dense_search(query, k=5):
    """Rank corpus documents by cosine similarity to the query.

    MATH: score(d) = (q . d) / (||q|| ||d||). Both q and d are L2-normalized
    here, so cosine similarity reduces to a plain dot product.
    """
    query_embedding = embedding_model.encode([query], normalize_embeddings=True)[0]
    similarities = doc_embeddings @ query_embedding
    ranked = np.argsort(-similarities)[:k]
    return [(doc_keys[i], float(similarities[i]), doc_texts[i]) for i in ranked]


query_numeric = "Which model has a vocabulary size of exactly 50257 tokens?"
print(f"Query: {query_numeric}\n")
for key, score, text in dense_search(query_numeric):
    print(f"  {score:.3f}  {key:18s} {text[:70]}")

**Failure mode 1 — exact terms get diluted.** The top-ranked document is
`bm25_doc`, which is about BM25's term-frequency formula and has nothing to do
with GPT-2. The document that actually answers the question, `gpt2_tokenizer`,
lands in second place. Both documents talk about "tokens" and "vocabulary" in
a generic sense, so their *embeddings* land close together in vector space —
but the embedding model has no special mechanism for the literal string
`50257`. A number that appears verbatim in the query and in exactly one
document is exactly the kind of signal a bag-of-words method is built for, and
exactly the kind of signal a dense embedding tends to wash out.

In [ ]:
query_multihop = "What tokenizer does GPT-2 use, and what algorithm does that tokenizer implement?"
print(f"Query: {query_multihop}\n")
for key, score, text in dense_search(query_multihop, k=3):
    print(f"  {score:.3f}  {key:18s} {text[:70]}")

top1_key, _, top1_text = dense_search(query_multihop, k=1)[0]
print("\nA naive top-1 retriever would hand the LLM only this chunk:")
print(f"  [{top1_key}] {top1_text}")
print("\nThat chunk answers 'which tokenizer' (BPE) but says nothing about")
print("what BPE itself does -- that explanation lives in a different document.")

**Failure mode 2 — multi-hop questions span chunks.** This time the *ranking*
is fine: `gpt2_tokenizer` and `bpe` are correctly the top two results. The
problem is `k`. The question has two parts ("which tokenizer" and "what does
that tokenizer's algorithm do"), and answering both parts fully requires
*both* documents. A naive pipeline with `k=1` retrieves only the first part of
the answer; even `k=2` only works because we got lucky that both relevant
chunks happen to rank highly for the *same* query.

Both failures share a root cause: **one ranking signal, one retrieval pass,
one fixed k.** Section 2 fixes failure mode 1 by adding a second, complementary
ranking signal. Section 4 fixes failure mode 2 by changing what gets sent to
the retriever in the first place.

## 2. Hybrid Search — BM25 + Dense + Reciprocal Rank Fusion

**BM25** is the classic sparse, keyword-based ranking function behind most
search engines before embeddings existed. For a query $q$ with terms $t$, it
scores a document $d$ as

$$\text{BM25}(q, d) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f(t, d) \cdot (k_1 + 1)}{f(t, d) + k_1 \cdot \left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)}$$

where $f(t, d)$ is how many times term $t$ appears in document $d$,
$\text{IDF}(t)$ (inverse document frequency) downweights terms that appear in
almost every document (like "the"), $|d|$ is the document's length, $\text{avgdl}$
is the average document length in the corpus, and $k_1$, $b$ are tuning
constants controlling term-frequency saturation and length normalization.
The intuition: a document that contains the query's exact words many times,
relative to how common those words are everywhere else, scores high — no
notion of "meaning" required.

Dense and BM25 fail on *different* queries, so combining their rankings tends
to be more robust than either alone. **Reciprocal Rank Fusion (RRF)** does
this combination using only each method's *rank order*, not its raw scores
(which live on incomparable scales — cosine similarity is bounded in $[-1, 1]$,
BM25 is an unbounded sum):

$$\text{RRF}(d) = \sum_{r \in \text{rankers}} \frac{1}{k_{\text{rrf}} + \text{rank}_r(d)}$$

Here $\text{rank}_r(d)$ is document $d$'s 1-indexed position in ranker $r$'s
full ranking (1 = best), and $k_{\text{rrf}}$ (commonly 60) is a constant that
flattens the curve so that, say, rank 1 vs rank 2 doesn't dominate rank 50 vs
rank 51 — every ranker's opinion contributes smoothly.

In [ ]:
def tokenize(text):
    """Lowercase and strip basic punctuation for BM25 term matching."""
    for ch in "'.,?":
        text = text.replace(ch, "")
    return text.lower().split()


bm25_index = BM25Okapi([tokenize(t) for t in doc_texts])


def bm25_search(query, k=5):
    """Rank corpus documents by BM25 score against the query's terms."""
    scores = bm25_index.get_scores(tokenize(query))
    ranked = np.argsort(-scores)[:k]
    return [(doc_keys[i], float(scores[i]), doc_texts[i]) for i in ranked]


print(f"Query: {query_numeric}\n")
for key, score, text in bm25_search(query_numeric):
    print(f"  {score:.3f}  {key:18s} {text[:70]}")

BM25 puts `gpt2_tokenizer` first by a wide margin, and `bm25_doc` doesn't even
make the top 5. The literal token `50257` appears in only one document, so its
inverse document frequency is high — exactly the signal BM25 is designed to
exploit, and exactly the signal the dense embedding in Section 1 missed.

In [ ]:
def reciprocal_rank_fusion(query, k=5, rrf_k=60):
    """Combine dense and BM25 rankings with Reciprocal Rank Fusion.

    MATH: RRF(d) = sum over rankers r of 1 / (rrf_k + rank_r(d)), where
    rank_r(d) is d's 1-indexed position in ranker r's full ranking.
    """
    dense_ranked = [key for key, _, _ in dense_search(query, k=len(doc_keys))]
    bm25_ranked = [key for key, _, _ in bm25_search(query, k=len(doc_keys))]

    dense_rank = {key: rank + 1 for rank, key in enumerate(dense_ranked)}
    bm25_rank = {key: rank + 1 for rank, key in enumerate(bm25_ranked)}

    rrf_scores = {
        key: 1 / (rrf_k + dense_rank[key]) + 1 / (rrf_k + bm25_rank[key])
        for key in doc_keys
    }
    ranked = sorted(doc_keys, key=lambda key: -rrf_scores[key])[:k]
    return [(key, rrf_scores[key], dense_rank[key], bm25_rank[key]) for key in ranked]


print(f"Query: {query_numeric}\n")
print(f"{'document':18s} {'rrf_score':>10s} {'dense_rank':>11s} {'bm25_rank':>10s}")
for key, rrf_score, d_rank, b_rank in reciprocal_rank_fusion(query_numeric):
    print(f"{key:18s} {rrf_score:>10.5f} {d_rank:>11d} {b_rank:>10d}")

`gpt2_tokenizer` is now ranked first overall — RRF recovered it using BM25's
rank-1 placement even though dense alone ranked it 2nd. Hybrid search fixed
failure mode 1.

Look closely at the scores, though: the top few RRF scores are very close
together (the gaps are in the fourth decimal place). RRF tells you the
*ordering* is right, but it doesn't give a confident, well-separated notion of
"how much better" the top result is. Section 3 adds a second pass that does.

## 3. Cross-Encoder Reranking

Everything so far has used a **bi-encoder**: the query and each document are
embedded *independently*, with no interaction between them, and compared with
a cheap similarity function (dot product). This is what makes dense search
scale — document embeddings are computed once, offline, and a query only needs
one new embedding plus a matrix multiply against millions of stored vectors.

A **cross-encoder** instead concatenates the query and a candidate document
into a single input — `[CLS] query [SEP] document [SEP]` — and runs the *whole
pair* through a transformer together, so every token of the query can attend to
every token of the document (and vice versa) before producing one relevance
score. This is far more accurate, because the model can directly check "does
this specific document contain what this specific query is asking for", but it
costs one full forward pass *per (query, document) pair* — far too slow to run
over an entire corpus. The standard pattern is therefore: use a cheap retriever
(Sections 1-2) to get a short candidate list, then rerank only that shortlist
with a cross-encoder.

In [ ]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def rerank(query, candidate_keys):
    """Score each (query, candidate document) pair jointly with a cross-encoder."""
    pairs = [(query, CORPUS[key]) for key in candidate_keys]
    scores = reranker.predict(pairs)
    order = np.argsort(-scores)
    return [(candidate_keys[i], float(scores[i])) for i in order]


candidates = [key for key, *_ in reciprocal_rank_fusion(query_numeric, k=5)]
print(f"RRF top-5 candidates: {candidates}\n")

print("Cross-encoder reranking:")
for key, score in rerank(query_numeric, candidates):
    print(f"  {score:7.3f}  {key:18s} {CORPUS[key][:65]}")

Compare the *margins*. RRF's top three scores were within about 0.001 of each
other — the fused ranking is correct but not confident. The cross-encoder gives
`gpt2_tokenizer` a score around +1.25 while every other candidate scores below
-10: a decisive, easily-thresholded separation. Because the cross-encoder reads
the query and document *together*, it can directly notice that the literal
substring `50257` appears in both — something neither the bi-encoder's
independent embeddings nor BM25's bag-of-words score can represent as cleanly.

## 4. Query Transformation — Multi-Query and HyDE

Sections 2-3 changed *how documents are ranked* for a fixed query. This section
changes *the query itself* before retrieval ever runs.

**Multi-query decomposition** asks an LLM to rewrite one (possibly compound)
question into several focused sub-questions, retrieves separately for each,
and unions the results. This is the direct fix for failure mode 2 from
Section 1: instead of hoping one query ranks every relevant chunk highly, each
sub-question only has to retrieve *its own* answer.

**HyDE (Hypothetical Document Embeddings)** asks an LLM to write a short
*hypothetical answer* to the query — a passage that, if true, would answer the
question — and embeds *that* instead of the raw query. The intuition: corpus
documents are written in "answer" register (statements of fact, definitions),
while user queries are written in "question" register. Embedding similarity is
sensitive to this register gap. A hypothetical answer, even if factually
imperfect, is written in the same register as the documents we're searching,
so its embedding tends to land closer to the real answer document.

In [ ]:
multi_query_llm = GenericFakeChatModel(messages=iter([
    AIMessage(content=json.dumps([
        "What tokenizer does GPT-2 use?",
        "What algorithm does the BPE tokenizer implement?",
    ]))
]))

decompose_prompt = f"Break this question into independent sub-questions as a JSON list: {query_multihop}"
sub_queries = json.loads(multi_query_llm.invoke(decompose_prompt).content)

print(f"Original question: {query_multihop}\n")
print("Sub-queries:")
for sub_query in sub_queries:
    print(f"  - {sub_query}")

print("\nRetrieval per sub-query (top-1 each):")
retrieved_keys = set()
for sub_query in sub_queries:
    key, score, text = dense_search(sub_query, k=1)[0]
    retrieved_keys.add(key)
    print(f"  '{sub_query}'\n    -> {key} ({score:.3f}): {text[:65]}")

single_key, _, _ = dense_search(query_multihop, k=1)[0]
print(f"\nSingle-query top-1 retrieval would only return: {{'{single_key}'}}")
print(f"Multi-query retrieval returns: {retrieved_keys}")

In [ ]:
layperson_query = "Why does ChatGPT split words into pieces instead of using whole words?"

hyde_llm = GenericFakeChatModel(messages=iter([
    AIMessage(content=(
        "Language models break text into smaller subword units rather than whole "
        "words because a fixed vocabulary cannot cover every possible word. "
        "Subword tokenization algorithms merge frequently occurring pairs of "
        "characters into larger tokens, producing an efficient vocabulary."
    ))
]))

print(f"Query: {layperson_query}\n")
print("Direct retrieval on the raw query:")
for key, score, text in dense_search(layperson_query, k=3):
    print(f"  {score:.3f}  {key:18s} {text[:65]}")

hypothetical_doc = hyde_llm.invoke(f"Write a short passage answering: {layperson_query}").content
print(f"\nHyDE hypothetical document:\n  {hypothetical_doc}")

hyde_embedding = embedding_model.encode([hypothetical_doc], normalize_embeddings=True)[0]
hyde_similarities = doc_embeddings @ hyde_embedding
print("\nRetrieval using the HyDE embedding instead of the raw query:")
for i in np.argsort(-hyde_similarities)[:3]:
    print(f"  {hyde_similarities[i]:.3f}  {doc_keys[i]:18s} {doc_texts[i][:65]}")

On the raw layperson query, `gpt2_tokenizer` edges out `bpe` (0.318 vs 0.299) —
but the question "why split into pieces" is really asking about the *BPE
algorithm itself*, not the GPT-2 trivia fact. The HyDE hypothetical passage,
written in the same definitional register as the corpus, flips this: `bpe`
moves to first place (0.384) ahead of `gpt2_tokenizer` (0.320). Neither ranking
is "wrong", but HyDE pulled the more directly explanatory document to the top
by closing the question/document phrasing gap.

That's the retrieval-quality toolkit: hybrid search for exact-term recall,
reranking for precision, query transformation for coverage and phrasing. The
last two sections turn this toolkit into something an *agent* can use.

## 5. Agentic RAG (Exercise)

Every retrieval so far has run *unconditionally*, exactly once, before
generation. **Agentic RAG** instead gives the LLM a `retrieve` tool — wrapping
the hybrid-search-plus-rerank pipeline from Sections 2-3 — inside the same
ReAct-style tool-calling loop from Topic 4 (`agent` node →
`tools_condition` → `tools` node → back to `agent`). The LLM decides *whether*
to call `retrieve` at all, and can call it *multiple times* if the first result
isn't enough — directly addressing failure mode 2 (multi-hop questions) without
needing a separate query-decomposition step.

The reference example below scripts a single retrieval call. The exercise asks
you to script the **two-call** case for the multi-hop GPT-2/BPE question from
Section 1.

In [ ]:
@tool
def retrieve(query: str) -> str:
    """Retrieve the single most relevant corpus document for a query, using hybrid search + reranking."""
    candidates = [key for key, *_ in reciprocal_rank_fusion(query, k=5)]
    top_key, _ = rerank(query, candidates)[0]
    return CORPUS[top_key]


class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


tool_node = ToolNode([retrieve])


def build_agent_graph(scripted_llm):
    """Wire a ReAct-style agent: agent -> (tool call?) -> tools -> agent -> ... -> END."""

    def call_model(state):
        return {"messages": [scripted_llm.invoke(state["messages"])]}

    builder = StateGraph(AgentState)
    builder.add_node("agent", call_model)
    builder.add_node("tools", tool_node)
    builder.add_edge(START, "agent")
    builder.add_conditional_edges("agent", tools_condition)
    builder.add_edge("tools", "agent")
    return builder.compile()


# Reference: a question that needs exactly one retrieval call
single_hop_llm = GenericFakeChatModel(messages=iter([
    AIMessage(content="", tool_calls=[
        {"name": "retrieve", "args": {"query": "What does tools_condition check for in LangGraph?"}, "id": "call_1"},
    ]),
    AIMessage(content="tools_condition checks the latest AIMessage for tool_calls and routes to the "
                      "tools node if present, or to END otherwise."),
]))

graph = build_agent_graph(single_hop_llm)
result = graph.invoke({"messages": [HumanMessage(content="What does tools_condition check for in LangGraph?")]})
for msg in result["messages"]:
    label = type(msg).__name__
    payload = getattr(msg, "tool_calls", None) or msg.content
    print(f"{label:12s} {payload}")

### Exercise — two-hop retrieval

Script a `GenericFakeChatModel` for the question
`"What tokenizer does GPT-2 use, and what algorithm does that tokenizer implement?"`
that makes the agent call `retrieve` **twice** before answering:

1. First `AIMessage`: `tool_calls=[{"name": "retrieve", "args": {"query": "What tokenizer does GPT-2 use?"}, "id": "call_1"}]`
2. Second `AIMessage`: `tool_calls=[{"name": "retrieve", "args": {"query": "What algorithm does the BPE tokenizer implement?"}, "id": "call_2"}]`
3. Third `AIMessage`: a final natural-language answer that combines *both*
   retrieved facts (which tokenizer GPT-2 uses, and what that tokenizer's
   algorithm actually does).

Build the graph with `build_agent_graph(...)`, invoke it on a `HumanMessage`
containing `query_multihop`, and print every message in `result["messages"]`
the same way the reference example does. Confirm you see two `ToolMessage`s —
one for each retrieval — before the final `AIMessage`.

In [ ]:
# TODO: script `multihop_llm` as described above, build the graph, run it on
# query_multihop, and print every message in result["messages"].

multihop_llm = None  # replace with a GenericFakeChatModel

## 6. GraphRAG (Exercise)

**GraphRAG** represents the corpus as a knowledge graph of `(entity, relation,
entity)` triples — in production, an LLM extracts these triples from each
document — and answers questions by *traversing the graph* rather than ranking
chunks. This is a different solution to the multi-hop problem: instead of
retrieving multiple chunks and hoping the LLM stitches them together, the
relationships between facts are made explicit and traversable up front.

The triples below encode (a subset of) the same facts as our corpus.

In [ ]:
TRIPLES = [
    ("GPT-2", "uses", "BPE_tokenizer"),
    ("BPE_tokenizer", "implements", "BPE_algorithm"),
    ("BPE_algorithm", "merges", "frequent_symbol_pairs"),
    ("LangGraph", "has_component", "checkpointer"),
    ("LangGraph", "has_component", "tools_condition"),
    ("checkpointer", "identified_by", "thread_id"),
    ("tools_condition", "checks", "AIMessage.tool_calls"),
    ("tools_condition", "routes_to", "tools_node"),
    ("tools_condition", "routes_to", "END"),
]

knowledge_graph = nx.DiGraph()
for head, relation, tail in TRIPLES:
    knowledge_graph.add_edge(head, tail, relation=relation)

print(f"Knowledge graph: {knowledge_graph.number_of_nodes()} nodes, {knowledge_graph.number_of_edges()} edges\n")


def describe_path(graph, path):
    """Render a node path as a chain of '<entity> --<relation>--> <entity>' steps."""
    return "\n  ".join(
        f"{u} --{graph[u][v]['relation']}--> {v}"
        for u, v in zip(path, path[1:])
    )


print("Worked example -- \"What does GPT-2's tokenizer actually do under the hood?\"")
path = nx.shortest_path(knowledge_graph, "GPT-2", "frequent_symbol_pairs")
print("  " + describe_path(knowledge_graph, path))

A single retrieved chunk (`gpt2_tokenizer`: "GPT-2 uses a BPE tokenizer...")
never mentions "merging frequent symbol pairs" — that's a 3-hop chain through
two other documents. Graph traversal follows the chain explicitly.

### Exercise — branching traversal

The `tools_condition` document says it "routes to the tools node if [tool
calls are present], **or to END otherwise**" — i.e. `tools_condition` has *two*
possible destinations. Use `nx.shortest_path` to find the path from
`"LangGraph"` to **each** of `"tools_node"` and `"END"`, and print both with
`describe_path`, the same way the worked example does.

In [ ]:
# TODO: for target in ["tools_node", "END"], compute and print
# nx.shortest_path(knowledge_graph, "LangGraph", target) using describe_path.

## Putting It All Together

| Stage | What changed | Failure it addresses |
|---|---|---|
| Naive RAG | dense embeddings, top-k | baseline (Section 1) |
| + Hybrid search (BM25 + RRF) | add a sparse, exact-term ranking signal | exact term/number matches diluted by dense embeddings |
| + Cross-encoder reranking | second pass with full query-document attention | narrow, low-confidence margins after fusion |
| + Multi-query | decompose the question before retrieving | multi-hop questions spanning several chunks |
| + HyDE | embed a hypothetical answer, not the raw question | question/document phrasing & register mismatch |
| Agentic RAG | LLM decides whether/how often to retrieve | multi-hop, without a separate decomposition step |
| GraphRAG | traverse explicit entity-relation edges | multi-hop chains the embedding space doesn't expose |

Each row is a strict addition on top of the previous one — a production system
would typically combine hybrid search + reranking as the default retriever,
and reach for query transformation, agentic loops, or graph traversal only for
the query patterns that need them.

## Where to Go Next

Topic 6 (Vector Databases) goes one level lower: how is `dense_search` above
actually implemented at scale? HNSW indexing, metadata filtering, and
*native* hybrid queries (the BM25 + dense + RRF pipeline from Section 2, but
running inside the database rather than in two separate Python data
structures) are the mechanics underneath everything built in this notebook.